In [1]:
from sympy import *
import copy

In [2]:
class T_symb_basis_elt:
    def __init__(self,str_rep,wght,ad_dict,parent):
        self.parent=parent
        self.str_rep=str_rep
        self.wght=wght
        
        self.vec_rep=[0]*len(parent.basis_strs)
        self.vec_rep[parent.basis_strs.index(str_rep)]=1
        
        self.ad_dict=copy.copy(ad_dict)
    
    def __eq__(self,other): 
        if type(other)==int:
            return False
        if self.parent!=other.parent: return False
        return self.vec_rep==other.vec_rep
        
    def __str__(self):
        return self.str_rep
    
    def __repr__(self):
        return self.str_rep
    
    def __lt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)<self.parent.basis.index(other)
    
    def __gt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)>self.parent.basis.index(other)
    
    def __le__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)<=self.parent.basis.index(other)

    def __ge__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)>=self.parent.basis.index(other)
    
    def __add__(self,other):
        if type(other) in [T_symb_elt,T_symb_basis_elt]:
            if self.parent!=other.parent: raise invalid_parent_exception 
            result=[other.vec_rep[i] for i in range(len(other.vec_rep))]
            result[self.parent.basis.index(self)]+=1
            return T_symb_elt(result,self.parent,vec_rep=True)
        if other==0:
            return self
        raise ValueError('Only objs of types T_symb_elt and T_symb_basis_elt can be added to an obj of type T_symb_elt')
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        result=[0]*len(self.parent.basis)
        result[self.parent.basis.index(self)]=-1
        return T_symb_elt(result,self.parent,vec_rep=True)
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,other):
        result=[0]*len(self.parent.basis)
        result[self.parent.basis.index(self)]=other
        return T_symb_elt(result,self.parent,vec_rep=True)
    
    def __rmul__(self,other):
        return(self*other)
        
    def ad(self,other,dual=False):
        return self.parent.elt(self.vec_rep,vec_rep=True).ad(other,dual)

In [3]:
class T_symb:
    def __init__(self,basis_strs,wght_list,ad_dict):
        '''basis_strs: list of strs repping basis elts
           wght_list: a list of wghts corresponding to basis_strs
           ad_dict: a dict with keys pairs of basis_strs and values coeff dicts
           '''
        self.basis_strs=copy.copy(basis_strs)
        self.basis=[T_symb_basis_elt(basis_strs[i],wght_list[i],{},self) # Set the ad_dicts in set_ad_dicts
                    for i in range(len(basis_strs))]
        self.ad_mats=[]
        self.ad_dict=copy.copy(ad_dict)
        self.set_ad_dict()
        
    def set_ad_dict(self,pickled_ad=False):
        # Set both the ad_dict of self and of the basis elements
        temp=copy.copy(self.ad_dict)
        for key in temp:
            A=key[0]
            B=key[1]
            
            A_elt=self.basis[self.basis_strs.index(A)]
            B_elt=self.basis[self.basis_strs.index(B)]
            
            C=self.elt(self.vec_rep_from_dict(temp[key]),vec_rep=True) # ad(A)(B) = C
            
            self.ad_dict[(A,B)]=C
            self.ad_dict[(B,A)]=-C
            
            A_elt.ad_dict[B]=C
            B_elt.ad_dict[A]=-C

    
    def set_ad_mats(self):
        for i in range(len(self.basis)):
            A=self.basis[i]
            r=Matrix([A.ad(B).vec_rep for B in self.basis])
            self.ad_mats.append(r.transpose())
            
    def Ad_mat(self,A):
        '''arg: a T_symb_elt or T_symb_basis_elt
           returns: the matrix rep of Ad(exp(A))'''
        ad_mat=zeros(len(self.basis))
        for i in range(len(self.basis)):
            ad_mat+=A.vec_rep[i]*self.ad_mats[i]
        return exp(ad_mat)
            
    
    def jacobi_test(self):
        '''returns: True if the Jacobi identity holds, False otherwise'''
        for A in self.basis:
            for B in self.basis:
                for C in self.basis:
                    t1=self.ad(A,self.ad(B,C))
                    t2=self.ad(self.ad(A,B),C)+self.ad(B,self.ad(A,C))
                    if t1!=t2: return False
        return True

#     To Do
#     def grading_test(self):
#         for A in self.basis:
#             for B in self.basis:
#                 t1=self.ad(A,B)
                
    
    def set_dual_ad_dict(self):
        # reset the dicts
        for A in self.basis:
            A.dual_ad_dict={B:self.elt([0]*len(self.basis),vec_rep=True) for B in self.basis_strs}
        # ad(A,B)=C ==> ad*(A,C*)-=B*
        for A in self.basis:
            for B in self.basis_strs:
                val=A.ad_dict[B]
                for i in range(len(self.basis)):
                    if val.vec_rep[i]!=0:
                        C=self.basis[i]
                        A.dual_ad_dict[str(C)]-=val.vec_rep[i]*self.basis[self.basis_strs.index(B)]
        
    
    def elt(self,coeff_dict={},vec_rep=False):
        if vec_rep is None: return(T_symb_elt([0]*len(self.basis),self,vec_rep=True))
        return T_symb_elt(coeff_dict,self,vec_rep)
    
    def sort_basis_tuple(self,basis_tuple):
        '''basis_tuple: a tuple of str_reps of T_symb_basis_elt objs
           returns: a tuple containing an rearrangement of basis_tuple of descending degree, 
           and the sign of the permutation (either -1 or 1)'''
        basis_list=list(basis_tuple)
        sorted_list=basis_list.copy()
        sorted_list.sort(key=lambda A:self.basis_strs.index(A))
        return(tuple(sorted_list),permutation_sign(basis_list,sorted_list))
    
    def vec_rep_from_dict(self,coeff_dict):
        '''coeff_dict: a dict representing an element of the algebra
           returns: the vector rep of the elt as a list'''
        result=[0]*len(self.basis)
        for A in coeff_dict:
            result[self.basis_strs.index(A)]=coeff_dict[A]
        return result
    
    def ad(self,t,c):
        '''t: a T_symb_elt object
           c: an object of type T_symb_basis_elt, T_symb_elt
           returns: ad(t,c)'''
        if type(t)==T_symb_elt or type(t)==T_symb_basis_elt:
            return t.ad(c)
        
        if type(t)==int and t==0:
            if type(c)==T_symb_elt or type(c)==T_symb_basis_elt:
                return c.parent.elt([0]*len(c.parent.basis),vec_rep=True)
        raise ValueError('The first argument of ad should be of type T_symb_elt or T_symb_basis_elt')

In [4]:
class T_symb_elt:
    
    def __init__(self,coeff_dict,parent, vec_rep=False):
        '''vec_rep: a list of length len(parent.basis) with integer entries'''
        self.parent=parent
        if vec_rep: self.vec_rep=copy.copy(coeff_dict)
        else: self.vec_rep=self.parent.vec_rep_from_dict(coeff_dict)
    
    def __str__(self):
        if self.vec_rep==[0]*len(self.parent.basis):
            return '0'
        
        result=''
        cntr=0
        while result=='':
            if self.vec_rep[cntr]!=0:
                if self.vec_rep[cntr]==1:
                    result=str(self.parent.basis[cntr])
                elif self.vec_rep[cntr]==-1:
                    result='-'+str(self.parent.basis[cntr])
                elif type(self.vec_rep[cntr])==Add:
                    result='('+str(self.vec_rep[cntr])+')*'+str(self.parent.basis[cntr])
                else:
                    result = str(self.vec_rep[cntr])+'*'+str(self.parent.basis[cntr])
            cntr+=1
        for i in range(cntr,len(self.parent.basis)):
            if self.vec_rep[i]==1:
                result+=' + '+str(self.parent.basis[i])
            elif self.vec_rep[i]==-1:
                result+=' - '+str(self.parent.basis[i])
            elif self.vec_rep[i]!=0:
                if type(self.vec_rep[i])==Add: c_str='('+str(self.vec_rep[i])+')*'
                else: c_str=str(self.vec_rep[i])
                result+=' + '+c_str+'*'+str(self.parent.basis[i])
        return result

    def __repr__(self):
        return str(self)
    
    def __eq__(self,other):
        if other==0 and type(other)==int:
            return self.vec_rep==[0]*(len(self.parent.basis))
        if not hasattr(other,'parent'): return False
        if self.parent!=other.parent: return False
        return self.vec_rep==other.vec_rep
    
    def __neg__(self):
        return(T_symb_elt([-A for A in self.vec_rep],self.parent,vec_rep=True))
    
    def __add__(self,other):
        if other==0:
            return self
        return T_symb_elt([self.vec_rep[i]+other.vec_rep[i] 
                           for i in range(len(self.parent.basis))],self.parent,vec_rep=True)   
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
            
    def __mul__(self,other):
        return T_symb_elt([other*A for A in self.vec_rep],self.parent,vec_rep=True)
    
    def __rmul__(self,other):
        return self*other
        
    def ad(self,t,dual=False):
        '''returns: a T_symb_elt object representing ad(self,t)
           if dual=True, the object returned represents ad(self,t*)'''
        P=self.parent
        
        if type(t)==T_symb_elt or type(t)==T_symb_basis_elt:
            if t.parent!=P: raise invalid_parent_exception
            
            if self==0 or t==0:
                return P.elt([0]*len(P.basis),vec_rep=True)

            result=P.elt([0]*len(P.basis),vec_rep=True)
            
            for i in range(len(P.basis)):
                if self.vec_rep[i]!=0:
                    for bs in P.basis[i].ad_dict:
                        j=P.basis_strs.index(bs)
                        result+=self.vec_rep[i]*t.vec_rep[j]*P.basis[i].ad_dict[bs]
            return result
        raise ValueError('ad must only be applied to objects T_symb_elt, T_symb_basis_elt')

In [5]:
class pro_T_symb:
    
    def __init__(self,T_symb,pro_number):
        self.orig_symb=T_symb
        self.v_coords=list(symbols('a1:'+str(pro_number+1)))
        self.v_ders=list(symbols('d1:'+str(pro_number+1)))
        self.basis_strs=[str(A) for A in self.v_ders]+T_symb.basis_strs
        self.basis=self.v_ders+T_symb.basis
        
    def elt(self,coeff_dict,vec_rep=False):
        return pro_T_symb_elt(coeff_dict,self,vec_rep)
    
    def phi(self,c1,c2):
        '''returns the product a[c1]*...*a[c2]'''
        if c1>c2:return 1
        n1=max(c1,1)
        n2=min(c2,len(self.v_coords))
        return prod(self.v_coords[n1-1:n2])
    
    def coordinatize(self,elt,B_inv):
        '''elt: An a pro_T_symb_elt
           B_inv: a matrix representing B^{-1} for B a matrix whose cols
                  form a basis for the pro_T_symb'''
        return list(B_inv*Matrix(elt.vec_rep))

In [142]:
class pro_T_symb_elt:
    
    def __init__(self,coeff_dict,parent,vec_rep=False): # coeff_dicts now might include v_ders
        self.parent=parent
        self.orig_symb=parent.orig_symb
        
        if vec_rep:
            self.vec_rep=copy.copy(coeff_dict)
            self.coeff_dict={}
            for i in range(len(parent.basis_strs)):
                if self.vec_rep[i]!=0:
                    self.coeff_dict[parent.basis_strs[i]]=self.vec_rep[i]
        
        else:
            self.coeff_dict=copy.copy(coeff_dict)
            self.vec_rep=[0]*len(parent.basis)
            for A in coeff_dict: self.vec_rep[parent.basis_strs.index(A)]=coeff_dict[A]
        
    def __str__(self):
        if self.coeff_dict=={}:
            return '0'
        
        result=''
        cntr=0
        while result=='':
            if cntr>len(self.vec_rep)-1: return('0')
            if self.vec_rep[cntr]!=0:
                if self.vec_rep[cntr]==1:
                    result=str(self.parent.basis[cntr])
                elif self.vec_rep[cntr]==-1:
                    result='-'+str(self.parent.basis[cntr])
                elif type(self.vec_rep[cntr])==Add:
                    result='('+str(self.vec_rep[cntr])+')*'+str(self.parent.basis[cntr])
                else:
                    result = str(self.vec_rep[cntr])+'*'+str(self.parent.basis[cntr])
            cntr+=1
        for i in range(cntr,len(self.parent.basis)):
            if self.vec_rep[i]==1:
                result+=' + '+str(self.parent.basis[i])
            elif self.vec_rep[i]==-1:
                result+=' - '+str(self.parent.basis[i])
            elif self.vec_rep[i]!=0:
                if type(self.vec_rep[i])==Add: c_str='('+str(self.vec_rep[i])+')*'
                else: c_str=str(self.vec_rep[i])
                result+=' + '+c_str+'*'+str(self.parent.basis[i])
        return result
    
    def __repr__(self):
        return str(self)
    
    def __eq__(self,other):
        if other==0 and type(other)==int:
            return self.vec_rep==[0]*(len(self.parent.basis))
        if not hasattr(other,'parent'): return False
        if self.parent!=other.parent: return False
        return self.vec_rep==other.vec_rep
    
    def __neg__(self):
        return(pro_T_symb_elt([-A for A in self.vec_rep],self.parent,vec_rep=True))
    
    def __add__(self,other):
        if other==0:
            return self
        return pro_T_symb_elt([self.vec_rep[i]+other.vec_rep[i] 
                           for i in range(len(self.parent.basis))],self.parent,vec_rep=True)   
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
            
    def __mul__(self,other):
        return pro_T_symb_elt([other*A for A in self.vec_rep],self.parent,vec_rep=True)
    
    def __rmul__(self,other):
        return self*other
    
    def simplify(self):
        self.coeff_dict={}
        for i in range(len(self.vec_rep)):
            bs=self.parent.basis_strs[i]
            coeff=simplify(self.vec_rep[i])
            self.coeff_dict[bs]=coeff
            self.vec_rep[i]=coeff
                
    def subs(self,subs_dict):
        '''Only accepts subs in dictionary format'''
        r=copy.copy(self)
        
        for key in r.coeff_dict:
            if hasattr(r.coeff_dict[key],'subs'):
                
                print('\n\n expr: ',r.coeff_dict[key])
                print('subs_dict: ',subs_dict)
                temp=r.coeff_dict[key].subs(subs_dict)
                r.coeff_dict[key]=temp
                r.vec_rep[r.parent.basis_strs.index(key)]=temp
        return r
        
    def ad(self,other):
        scd=self.coeff_dict
        ocd=other.coeff_dict
        v_strs=[str(A) for A in self.parent.v_ders]
        temp1=self.orig_symb.elt({A:scd[A] for A in scd if A not in v_strs})
        temp2=self.orig_symb.elt({A:ocd[A] for A in ocd if A not in v_strs})
        r_vec=[0]*len(v_strs)+copy.copy(temp1.ad(temp2).vec_rep)
        
        for i in range(len(self.parent.v_ders)):
            d=str(self.parent.v_ders[i])
            a=self.parent.v_coords[i]
            if d in scd:
                for key in ocd:
                    c=diff(ocd[key],a)*scd[d]
                    if c!=0:
                        j=self.parent.basis_strs.index(key)
                        r_vec[j]=r_vec[j]+c
            if d in ocd:
                for key in scd:
                    c=diff(scd[key],a)*ocd[d]
                    if c!=0:
                        j=self.parent.basis_strs.index(key)
                        r_vec[j]=r_vec[j]-c
        return pro_T_symb_elt(r_vec,self.parent,vec_rep=True)

## Prolongation Computations

### Free Symbols

In [143]:
F3_ad_dict={('X1','X2'):{'X3':1},('X1','X3'):{'X4':1},('X2','X3'):{'X5':1}}


F4_ad_dict=copy.copy(F3_ad_dict)
F4_ad_dict.update({('X1','X4'):{'X6':1},('X1','X5'):{'X7':1},('X2','X4'):{'X7':1},
                             ('X2','X5'):{'X8':1}})

F5_ad_dict=copy.copy(F4_ad_dict)
F5_ad_dict.update({('X1','X6'):{'X9':1},('X1','X7'):{'X10':1},('X1','X8'):{'X11':1},
                              ('X2','X6'):{'X12':1},('X2','X7'):{'X13':1},('X2','X8'):{'X14':1},
                              ('X3','X4'):{'X10':1,'X12':-1},('X3','X5'):{'X11':1,'X13':1}})

In [144]:
F3_str_list=['X1','X2','X3','X4','X5']
F4_str_list=F3_str_list+['X6','X7','X8']
F5_str_list=F4_str_list+['X9','X10','X11','X12','X13','X14']

F3_wght_list=[-1,-1,-2,-3,-3]
F4_wght_list=F3_wght_list+[-4]*(len(F4_str_list)-len(F3_str_list))
F5_wght_list=F4_wght_list+[-5]*(len(F5_str_list)-len(F4_str_list))

In [145]:
F3=T_symb(F3_str_list,F3_wght_list,F3_ad_dict)
F4=T_symb(F4_str_list,F4_wght_list,F4_ad_dict)
F5=T_symb(F5_str_list,F5_wght_list,F5_ad_dict)

Let $D$ be a distribution which is not of maximal class. Then for some $1\leq \ell \leq n-5$, there is a Cauchy characteristic of $(\text{pr}^\ell D)^{-2\ell-2}$ in $\text{pr}^\ell D$

The manifold $\text{pr}^\ell M$ has a frame $$Y_{1-\ell}=\partial_{a^\ell}, Y_{2-\ell} = ,\ldots, Y_1 = X_1+a^1X_2, Y_2=X_2,\ldots Y_n=X_n$$

$$Y_i = \sum_{\alpha=1}^{1-i}\varphi_{\alpha+2}^{2-i}\partial_{a^\alpha} + \varphi_{2}^{2-i}(X_1+a^1 X_2)$$

where $\varphi_{\beta}^\gamma = a^\beta\cdot a^{\beta+1}\cdots a^{\gamma-1}\cdot a^{\gamma}$

## F4 (n=8), l=4 (corank 1)

In [146]:
l=4
n=8

P4F4=pro_T_symb(F4,l)
a=P4F4.v_coords
d=[P4F4.elt({str(A):1}) for A in P4F4.v_ders]

X1=P4F4.elt({'X1':1})
X2=P4F4.elt({'X2':1})

Y={} # This will represent our frame; we won't use a list so that we can have negative indices
for i in range(2,n+1):
    Y[i]=P4F4.elt({'X'+str(i):1})
Y[1]=(X1+a[0]*X2)
for i in range(l-1):
    Y[-i]=d[i]+a[i+1]*Y[1-i]
Y[1-l]=d[l-1]

In [ ]:
list(Y.keys())

In [174]:
## Osculating Flag

W={}

# -1
W[-3]=Y[-3]
W[-2]=Y[-2]

#-2
W[-1]=Y[-3].ad(W[-2]) # = Y[-1]

#-3
W[0]=Y[-2].ad(W[-1]) # = Y[0]

#-4
W[1]=Y[-2].ad(W[0]) # = Y[1]

#-5
W[2]=Y[-2].ad(W[1]) # = Y[2]

#-6
W[3]=Y[-2].ad(W[2])# = Y[3]

#-7
W[4]=Y[-2].ad(W[3])# = Y[4] + a1*Y[5]

#-8
W[5]=Y[-2].ad(W[4])

#-9
W[6]=Y[-2].ad(W[5])
W[6].simplify()

#-10
W[7]=Y[-2].ad(W[6])

#-11
W[8]=Y[-3].ad(W[7])
W[8].simplify()

In [ ]:
for i in range(-3,9):
    print('--------------------- W',[i],'---------------------')
    for key in W[i].coeff_dict:
        print(key,'-->',W[i].coeff_dict[key])

In [148]:
## Let's compute an adapted frame

Z={}

# -1
Z[-3]=Y[-3]
Z[-2]=Y[-2]

#-2
Z[-1]=Y[-3].ad(Z[-2]) # = Y[-1]

#-3
Z[0]=Y[-2].ad(Z[-1]) # = Y[0]

#-4
Z[1]=Y[-2].ad(Z[0])*(1/a[3]) # = Y[1]

#-5
Z[2]=Y[-2].ad(Z[1])*(1/(a[2]*a[3])) # = Y[2]

#-6
Z[3]=Y[-2].ad(Z[2])*(1/(a[3]*a[2]*a[1]))# = Y[3]

#-7
Z[4]=Y[-2].ad(Z[3])*(1/(a[3]*a[2]*a[1]))# = Y[4] + a1*Y[5]

#-8
Z[5]=Y[-2].ad(Z[4])*(1/(a[3]*a[2]))

#-9
Z[6]=Y[-2].ad(Z[5])*(1/a[3])
Z[6].simplify()

#-10
Z[7]=Y[-2].ad(Z[6])

#-11
Z[8]=Y[-3].ad(Z[7])*(1/a[2])
Z[8].simplify()

In [149]:
Bonus=Y[-2].ad(Z[7])*(1/a[3])
Bonus.simplify()

In [ ]:
for i in range(-3,9):
    print('\nZ',[i],'=',Z[i])
    print('Y',[i],'=',Y[i])
    
print('\nBonus =',Bonus)

In [ ]:
P4F4_YB_inv=transpose(Matrix([Y[i].vec_rep for i in range(-3,9)])).inv()
Z_coord_list=[]
for i in range(-3,9):
    Z_coord_list.append(P4F4.coordinatize(Z[i],P4F4_YB_inv))
    print('Z',[i],'=',P4F4.coordinatize(Z[i],P4F4_YB_inv))
Z_mat=Matrix(Z_coord_list).transpose()

In [152]:
B0=Z_mat.col(shape(Z_mat)[1]-1)
B1=Matrix(P4F4.coordinatize(Bonus,P4F4_YB_inv))
B2=Z_mat.col(shape(Z_mat)[1]-2)

LI_Mat=Matrix([list(B0),list(B1),list(B2)]).transpose()

In [153]:
LI_Mat=LI_Mat1.col_insert(shape(LI_Mat1)[1],zeros(shape(LI_Mat1)[0],1))

In [154]:
x,y,z=symbols('x,y,z')
sols=solve_linear_system(LI_Mat1,x,y,z)

In [ ]:
den1=(21*a[1] + 40*a[2]*a[3])

c0=sols[x]*den1/z
c1=sols[y]*den1/z

CC_temp=c0*Y[-3]*(1/a[2])+c1*Y[-2]*(1/a[3])

den2=a[2]*a[3]
CC=CC_temp*den2

temp=CC.ad(Z[7])+den2*c2*Z[7]
temp.simplify()
print('CC verification:',temp==temp.parent.elt({}))
print('\nCC =',CC)

# # Z[8] = ad(Y[-3])(Z[7])/a[2]
# # Bonus = ad(Y[-2])(Z[7])/a[3]

In [ ]:
A=Z_mat[l-1:shape(Z_mat)[1],l-1:shape(Z_mat)[1]]
A

In [ ]:
det(A)

In [157]:
# Sanity check
for i in range(7):
    if Z[-3].ad(Z[i])!=0: print('Failure at i =',i)

## F5 (n=14), l=10 (corank 1)

In [161]:
l=10
n=14

P10F5=pro_T_symb(F5,l)
a=P10F5.v_coords
d=[P10F5.elt({str(A):1}) for A in P10F5.v_ders]

X1=P10F5.elt({'X1':1})
X2=P10F5.elt({'X2':1})

Y={} # This will represent our frame; we won't use a list so that we can have negative indices
for i in range(2,n+1):
    Y[i]=P10F5.elt({'X'+str(i):1})
Y[1]=(X1+a[0]*X2)
for i in range(l-1):
    Y[-i]=d[i]+a[i+1]*Y[1-i]
Y[1-l]=d[l-1]

In [162]:
# list(Y.keys()) #-9 to 14

In [163]:
## Let's compute an adapted frame

Z={}

# -1
Z[-9]=Y[-9]
Z[-8]=Y[-8]
#-2
Z[-7]=Y[-9].ad(Z[-8]) 
#-3
Z[-6]=Y[-8].ad(Z[-7]) 
#-4
Z[-5]=Y[-8].ad(Z[-6])*(1/(a[9]))
#-5
Z[-4]=Y[-8].ad(Z[-5])*(1/(a[9]*a[8])) 
#-6
Z[-3]=Y[-8].ad(Z[-4])*(1/(a[9]*a[8]*a[7]))
#-7
Z[-2]=Y[-8].ad(Z[-3])*(1/(a[9]*a[8]*a[7]*a[6]))
#-8
Z[-1]=Y[-8].ad(Z[-2])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]))
#-9
Z[0]=Y[-8].ad(Z[-1])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]))
Z[0].simplify()
#-10
Z[1]=Y[-8].ad(Z[0])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]*a[3]))
Z[1].simplify()
#-11
Z[2]=Y[-8].ad(Z[1])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]*a[3]*a[2]))
Z[2].simplify()
#-12
Z[3]=Y[-8].ad(Z[2])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]*a[3]*a[2]*a[1]))
Z[3].simplify()
#-13
Z[4]=Y[-8].ad(Z[3])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]*a[3]*a[2]*a[1]))
Z[4].simplify()
#-14
Z[5]=Y[-8].ad(Z[4])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]*a[3]*a[2]))
Z[5].simplify()
#-15
Z[6]=Y[-8].ad(Z[5])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]*a[3]))
Z[6].simplify()
#-16
Z[7]=Y[-8].ad(Z[6])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]*a[4]))
Z[7].simplify()
#-17
Z[8]=Y[-8].ad(Z[7])*(1/(a[9]*a[8]*a[7]*a[6]*a[5]))
Z[8].simplify()
#-18
Z[9]=Y[-8].ad(Z[8])*(1/(a[9]*a[8]*a[7]*a[6]))
Z[9].simplify()
#-19
Z[10]=Y[-8].ad(Z[9])*(1/(a[9]*a[8]*a[7]))
Z[10].simplify()
#-20
Z[11]=Y[-8].ad(Z[10])*(1/(a[9]*a[8]))
Z[11].simplify()
#-21
Z[12]=Y[-8].ad(Z[11])*(1/(a[9]))
Z[12].simplify()
#-22
Z[13]=Y[-8].ad(Z[12])
Z[13].simplify()
#-23
Z[14]=Y[-9].ad(Z[13])
Z[14].simplify()


# # For computing the CC
Bonus=Y[-8].ad(Z[13])
Bonus.simplify()

In [ ]:
for i in range(-7,4):
    if Z[i]!=Y[i]: print('Failure at',i)

for i in range(13,15):
    print('\nZ',[i],'=',Z[i])
    print('Y',[i],'=',Y[i])

# for i in range(4,15):
#     print('\nZ',[i],'=',Z[i])
#     print('Y',[i],'=',Y[i])
    
# print('\nBonus =',Bonus)

In [ ]:
P10F5_YB_inv=transpose(Matrix([Y[i].vec_rep for i in range(-9,15)])).inv()
Z_coord_list=[]
for i in range(-9,15):
    Z_coord_list.append(P10F5.coordinatize(Z[i],P10F5_YB_inv))
    print('Z',[i],'=',P10F5.coordinatize(Z[i],P10F5_YB_inv))
Z_mat=Matrix(Z_coord_list).transpose()

In [172]:
LI_Mat=Z_mat[0:shape(Z_mat)[1],shape(Z_mat)[0]-2:shape(Z_mat)[0]]
# display(LI_Mat)
LI_Mat=LI_Mat.col_insert(shape(LI_Mat)[1],Matrix(P10F5.coordinatize(Bonus,P10F5_YB_inv)))
# display(LI_Mat)
LI_Mat=LI_Mat[shape(LI_Mat)[0]-2:shape(LI_Mat)[0],0:shape(LI_Mat)[1]]

In [ ]:
LI_Mat
A1=LI_Mat.col(1)
A2=LI_Mat.col(2)
A0=LI_Mat.col(0)

In [ ]:
LI_Mat=LI_Mat.col_insert(shape(LI_Mat)[1],zeros(shape(LI_Mat)[0],1))
display(LI_Mat)

In [ ]:
x,y,z=symbols('x,y,z')
sols=solve_linear_system(LI_Mat,x,y,z)

In [ ]:
c0=sols[x]/z*9*a[1]**2
c1=sols[y]/z*9*a[1]**2
c2=9*a[1]**2
temp=simplify(c0*A0+c1*A1+c2*A2) # = 0
print('c1 =',c1,'\nc2 =',c2)

In [ ]:
CC=c1*Y[-3]*a[3]+c2*Y[-2]*a[2]
temp=CC.ad(Z[7])+c0*Z[7]*a[2]*a[3]
temp.simplify()
temp

In [ ]:
CC

In [ ]:
A=Z_mat[l-1:shape(Z_mat)[1],l-1:shape(Z_mat)[1]]
A

In [ ]:
det(A)

In [ ]:
# Sanity check
for i in range(13):
    if Z[-9].ad(Z[i])!=0: print('Failure at i =',i)

## F4 (n=8), l=3 (corank 2)

In [15]:
l=3
n=8

P3F4=pro_T_symb(F4,l)
a=P3F4.v_coords
d=[P3F4.elt({str(A):1}) for A in P3F4.v_ders]

X1=P3F4.elt({'X1':1})
X2=P3F4.elt({'X2':1})

Y={} # This will represent our frame; we won't use a list so that we can have negative indices
for i in range(2,n+1):
    Y[i]=P3F4.elt({'X'+str(i):1})
Y[1]=(X1+a[0]*X2)
for i in range(l-1):
    Y[-i]=d[i]+a[i+1]*Y[1-i]
Y[1-l]=d[l-1]

In [16]:
## Let's compute an adapted frame

Z={}

# -1
Z[-2]=Y[-2]
Z[-1]=Y[-1]

#-2
Z[0]=Y[-2].ad(Y[-1]) # = Y[0]

#-3
Z[1]=Y[-1].ad(Y[0]) # = Y[1]

#-4
Z[2]=Y[-1].ad(Z[1])*(1/a[2]) # = Y[2]

#-5
Z[3]=Y[-1].ad(Z[2])*(1/a[2]/a[1]) # = Y[3]

#-6
Z[4]=Y[-1].ad(Z[3])*(1/a[2]/a[1]) # = Y[4] + a1*Y[5]

#-7
Z[5]=Y[-1].ad(Z[4])*(1/a[2])

#-8
Z[6]=Y[-1].ad(Z[5])

# We don't want -8 to a a CC in -1

#-9
Z[7]=Y[-1].ad(Z[6])*(Rational(1,5)/a[2])
Z[7].simplify()
Z[8]=Y[-2].ad(Z[6])*(Rational(1,3)/a[1])

In [ ]:
for i in range(-2,9):
    print('Z',[i],'=',Z[i])

In [18]:
P3F4_YB_inv=transpose(Matrix([Y[i].vec_rep for i in range(-2,9)])).inv()

In [ ]:
Z_coord_list=[]
for i in range(-2,9):
    Z_coord_list.append(P3F4.coordinatize(Z[i],P3F4_YB_inv))
    print('Z',[i],'=',P3F4.coordinatize(Z[i],P3F4_YB_inv))
Z_mat=Matrix(Z_coord_list).transpose()

In [ ]:
Z_mat.det()

In [ ]:
Z_mat

In [442]:
A=Z_mat[3:shape(Z_mat)[1],3:shape(Z_mat)[1]]

In [ ]:
A

In [438]:
# Sanity check
for i in range(6):
    if Z[-2].ad(Z[i])!=0: print('Failure at i =',i)

## F5 (n=14) l=8 (Corank 3)
No CC of $(\text{pr}^8D)^{-18}$, which has rank 20

In [358]:
l=8
n=14

P8F5=pro_T_symb(F5,l)
a=P8F5.v_coords
d=[P8F5.elt({str(A):1}) for A in P8F5.v_ders]

X1=P8F5.elt({'X1':1})
X2=P8F5.elt({'X2':1})

Y={} # This will represent our frame; we won't use a list so that we can have negative indices
for i in range(2,n+1):
    Y[i]=P8F5.elt({'X'+str(i):1})
Y[1]=(X1+a[0]*X2)
for i in range(l-1):
    Y[-i]=d[i]+a[i+1]*Y[1-i]
Y[1-l]=d[l-1]

In [ ]:
print(sorted(list(Y.keys())))

In [360]:
## Let's compute an osculating flag

Z[-7]=Y[-7]
Z[-6]=Y[-6]
Z[-5]=Z[-7].ad(Z[-6]) # = Y[-5]
Z[-4]=Z[-6].ad(Z[-5]) # = Y[-4]
Z[-3]=Z[-6].ad(Z[-4])*(1/a[7]) # = Y[-3]
Z[-2]=Z[-6].ad(Z[-3])*(1/(a[6]*a[7]))# = Y[-2]
Z[-1]=Z[-6].ad(Z[-2])*(1/(a[5]*a[6]*a[7]))# !=Y[-1]
Z[0]=Z[-6].ad(Z[-1])*(1/(a[4]*a[5]*a[6]*a[7])) # = Y[0]
Z[1]=Z[-6].ad(Z[0])*(1/(a[3]*a[4]*a[5]*a[6]*a[7])) # = Y[0]
Z[2]=Z[-6].ad(Z[1])*(1/(a[2]*a[3]*a[4]*a[5]*a[6]*a[7])) # = Y[1]
Z[3]=Z[-6].ad(Z[2])*(1/(a[1]*a[2]*a[3]*a[4]*a[5]*a[6]*a[7])) # = Y[2]
Z[4]=Z[-6].ad(Z[3])*(1/(a[1]*a[2]*a[3]*a[4]*a[5]*a[6]*a[7])) # = Y[3]
Z[5]=Z[-6].ad(Z[4])*(1/(a[2]*a[3]*a[4]*a[5]*a[6]*a[7])) # = Y[4] + a1*Y[5]
Z[6]=Z[-6].ad(Z[5])*(1/(a[3]*a[4]*a[5]*a[6]*a[7]))
Z[6].simplify()
Z[7]=Z[-6].ad(Z[6])*(1/(a[4]*a[5]*a[6]*a[7]))
Z[7].simplify()
Z[8]=Z[-6].ad(Z[7])*(1/(a[5]*a[6]*a[7]))
Z[8].simplify()
Z[9]=Z[-6].ad(Z[8])*(1/(a[6]*a[7]))
Z[9].simplify()
Z[10]=Z[-6].ad(Z[9])*(1/(a[7]))
Z[10].simplify()
Z[11]=Z[-6].ad(Z[10])
Z[11].simplify()
# Z[12]=Z[-6].ad(Z[11])
# Z[12].simplify()
# Z[13]=Z[-6].ad(Z[12])
# Z[13].simplify()
# ## This shouldn't have a CC in Z[-8], Z[-7]

# Z[13]=Z[-7].ad(Z[12])*(1/a[8])
# Z[13].simplify()
# Z[14]=Z[-8].ad(Z[12])
# Z[14].simplify()

# ## We want to show that all these are generically linearly independent

In [362]:
P8F5_YB_inv=transpose(Matrix([Y[i].vec_rep for i in range(-7,15)])).inv()

In [ ]:
Z_coord_list=[]
for i in range(-7,12):
    Z_coord_list.append(P8F5.coordinatize(Z[i],P8F5_YB_inv))
    print('Z',[i],'=',P8F5.coordinatize(Z[i],P8F5_YB_inv))
Z_mat=Matrix(Z_coord_list).transpose()

In [ ]:
for i in range(-7,12):
    print('\n\n')
    v=P8F5.coordinatize(Z[i],P8F5_YB_inv)
    print('---------------Z[',i,']---------------')
    for j in range(len(v)):
        if v[j]!=0: print('    Y[',j-l+1,']:',v[j].subs(subs_dict))

In [23]:
## Sanity check

for i in range(-6,12):
    r=Z[-8].ad(Z[i])
    if r!=0:
        print(r)

In [75]:
P8F5_FM=zeros(len(P8F5.basis))
for i in range(len(P8F5.basis_strs)):
    Z_elt=Z[i-l+1]
    for j in range(len(P8F5.basis)):
        b_elt=P8F5.basis_strs[j]
        if b_elt in Z_elt.coeff_dict:
            P8F5_FM[i,j]=Z_elt.coeff_dict[b_elt]

In [121]:
subs_dict={A:1 for A in a}
subs_dict[a[0]]=0 # O
# subs_dict[a[1]]=0 # X
# subs_dict[a[2]]=0 # X
# subs_dict[a[3]]=0 # X
# subs_dict[a[4]]=0 # X
# subs_dict[a[5]]=0 # X
# subs_dict[a[6]]=0 # X
# subs_dict[a[7]]=0 # X
subs_dict[a[8]]=0 # O
S0=P9F5_FM.subs(subs_dict)

In [ ]:
S0.det()

In [ ]:
P9F5_FM.subs(subs_dict)

In [ ]:
P9F5_FM.subs({a[0]:0,a[8]:0})

## F5 (n=14) l=9 (Corank 2)
No CC of $(\text{pr}^{9}D)^{-20}$, which has rank 21

In [368]:
l=9
n=14

P9F5=pro_T_symb(F5,l)
a=P9F5.v_coords
d=[P9F5.elt({str(A):1}) for A in P9F5.v_ders]

X1=P9F5.elt({'X1':1})
X2=P9F5.elt({'X2':1})

Y={} # This will represent our frame; we won't use a list so that we can have negative indices
for i in range(2,n+1):
    Y[i]=P9F5.elt({'X'+str(i):1})
Y[1]=(X1+a[0]*X2)
for i in range(l-1):
    Y[-i]=d[i]+a[i+1]*Y[1-i]
Y[1-l]=d[l-1]

In [ ]:
print(sorted(list(Y.keys())))

In [371]:
## Let's compute an osculating flag

Z[-8]=Y[-8]
Z[-7]=Y[-7]
Z[-6]=Z[-8].ad(Z[-7]) # = Y[-6]
Z[-5]=Z[-7].ad(Z[-6]) # = Y[-5]
Z[-4]=Z[-7].ad(Z[-5])*(1/a[8]) # = Y[-4]
Z[-3]=Z[-7].ad(Z[-4])*(1/a[8]/a[7]) # = Y[-3]
Z[-2]=Z[-7].ad(Z[-3])*(1/(a[6]*a[7]*a[8])) # = Y[-2]
Z[-1]=Z[-7].ad(Z[-2])*(1/(a[5]*a[6]*a[7]*a[8])) # = Y[-1]
Z[0]=Z[-7].ad(Z[-1])*(1/(a[4]*a[5]*a[6]*a[7]*a[8])) # = Y[0]
Z[1]=Z[-7].ad(Z[0])*(1/(a[3]*a[4]*a[5]*a[6]*a[7]*a[8])) # = Y[1]
Z[2]=Z[-7].ad(Z[1])*(1/(a[2]*a[3]*a[4]*a[5]*a[6]*a[7]*a[8])) # = Y[2]
Z[3]=Z[-7].ad(Z[2])*(1/(a[1]*a[2]*a[3]*a[4]*a[5]*a[6]*a[7]*a[8])) # = Y[3]
Z[4]=Z[-7].ad(Z[3])*(1/(a[1]*a[2]*a[3]*a[4]*a[5]*a[6]*a[7]*a[8])) # = Y[4] + a1*Y[5]
Z[5]=Z[-7].ad(Z[4])*(1/(a[2]*a[3]*a[4]*a[5]*a[6]*a[7]*a[8]))
Z[6]=Z[-7].ad(Z[5])*(1/(a[3]*a[4]*a[5]*a[6]*a[7]*a[8]))
Z[6].simplify()
Z[7]=Z[-7].ad(Z[6])*(1/(a[4]*a[5]*a[6]*a[7]*a[8]))
Z[7].simplify()
Z[8]=Z[-7].ad(Z[7])*(1/(a[5]*a[6]*a[7]*a[8]))
Z[8].simplify()
Z[9]=Z[-7].ad(Z[8])*(1/(a[6]*a[7]*a[8]))
Z[9].simplify()
Z[10]=Z[-7].ad(Z[9])*(1/(a[7]*a[8]))
Z[10].simplify()
Z[11]=Z[-7].ad(Z[10])*(1/(a[8]))
Z[11].simplify()
Z[12]=Z[-7].ad(Z[11])
Z[12].simplify()
## This shouldn't have a CC in Z[-8], Z[-7]

Z[13]=Z[-7].ad(Z[12])*(1/a[8])
Z[13].simplify()
Z[14]=Z[-8].ad(Z[12])
Z[14].simplify()

## We want to show that all these are generically linearly independent

In [372]:
## Sanity check

for i in range(-6,12):
    r=Z[-8].ad(Z[i])
    if r!=0:
        print(r)

In [410]:
P9F5_YB_inv=transpose(Matrix([Y[i].vec_rep for i in range(-8,15)])).inv()

In [ ]:
Z_coord_list=[]
for i in range(-8,15):
    Z_coord_list.append(P9F5.coordinatize(Z[i],P9F5_YB_inv))
    print('len(Z',[i],') =',len(P9F5.coordinatize(Z[i],P9F5_YB_inv)))
Z_mat=Matrix(Z_coord_list).transpose() # In basis of Yi

In [ ]:
A

In [ ]:
print(shape(Z_mat))
A=Z_mat[9:-1,9:-1] # Bottom left nxn submatrix

In [ ]:
subs_dict={L:1 for L in a}
subs_dict[a[0]]=0 # O
# subs_dict[a[1]]=0 # X
# subs_dict[a[2]]=0 # X
# subs_dict[a[3]]=0 # X
# subs_dict[a[4]]=0 # X
# subs_dict[a[5]]=0 # X
# subs_dict[a[6]]=0 # X
# subs_dict[a[7]]=0 # X
subs_dict[a[8]]=0 # O

A.subs(subs_dict)

In [425]:
growth=[2,3,5,8,14]
diag_blocks=[]
super_diag_blocks=[]
sub_diag_blocks=[]
for i in range(len(growth)):
    if i==0: s0=0
    else: s0=growth[i-1]
    s1=growth[i]
    diag_blocks.append(A[s0:s1,s0:s1])
    for j in range(i+1,len(growth)):
        r0=growth[j-1]
        r1=growth[j]
        super_diag_blocks.append(A[s0:s1,r0:r1])
        sub_diag_blocks.append(A[r0:r1,s0:s1])

In [ ]:
for S in diag_blocks: 
    display(S)

In [75]:
P9F5_FM=zeros(len(P9F5.basis)) # In basis of di and Xj
for i in range(len(P9F5.basis_strs)):
    Z_elt=Z[i-l+1]
    for j in range(len(P9F5.basis)):
        b_elt=P9F5.basis_strs[j]
        if b_elt in Z_elt.coeff_dict:
            P9F5_FM[i,j]=Z_elt.coeff_dict[b_elt]

In [121]:
subs_dict={A:1 for A in a}
subs_dict[a[0]]=0 # O
# subs_dict[a[1]]=0 # X
# subs_dict[a[2]]=0 # X
# subs_dict[a[3]]=0 # X
# subs_dict[a[4]]=0 # X
# subs_dict[a[5]]=0 # X
# subs_dict[a[6]]=0 # X
# subs_dict[a[7]]=0 # X
subs_dict[a[8]]=0 # O
S0=P9F5_FM.subs(subs_dict)

In [ ]:
S0.det()

In [ ]:
P9F5_FM.subs(subs_dict)

In [ ]:
P9F5_FM.subs({a[0]:0,a[8]:0})

### Experimentation

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]
for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    print(i,'-->',Z[0].parent.elt(temp),'\n\n')

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]
for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    print(i,'-->',Z[0].parent.elt(temp),'\n\n')

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]
for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    print(i,'-->',Z[i].coeff_dict['X9'],'\n\n')

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]


X_str='X10'

for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    if X_str in Z[i].coeff_dict:
        print(i,'-->',Z[i].coeff_dict['X10'],'\n\n')

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]

X_str='X14'
hit_list=[2,3,4,5]
pass_list=[]
subs_dict={a[i-1]:0 for i in hit_list}
subs_dict.update({a[i-1]:1 for i in pass_list})



for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    if X_str in Z[i].coeff_dict:
        print(i,'-->',Z[i].coeff_dict[X_str].subs(subs_dict),'\n\n')

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]

X_str='X14'
hit_list=[1,2,9]
pass_list=[]
subs_dict={a[i-1]:0 for i in hit_list}
subs_dict.update({a[i-1]:1 for i in pass_list})



for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    if X_str in Z[i].coeff_dict:
        print(i,'-->',Z[i].coeff_dict[X_str].subs(subs_dict),'\n\n')

In [ ]:
i_list=list(range(-8,15))
pb=Z[0].parent.basis
pbs=Z[0].parent.basis_strs
pbsi=Z[0].parent.basis_strs[i_list[0]+l-1:i_list[-1]+l]

X_str='X14'
hit_list=[2,3,4,5]
pass_list=[]
subs_dict={a[i-1]:0 for i in hit_list}
subs_dict.update({a[i-1]:1 for i in pass_list})



for i in i_list:
    temp={A:Z[i].coeff_dict[A] for A in pbsi if A in Z[i].coeff_dict}
    if X_str in Z[i].coeff_dict:
        print(i,'-->',Z[i].coeff_dict[X_str].subs(subs_dict),'\n\n')